# DB에 유저 정보를 저장

In [1]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_prompt_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [3]:
from typing import Dict, List # 타이핑 형식 검증 용
from langchain_core.chat_history import InMemoryChatMessageHistory # 대화 메시지를 메모리에 저장하고 관리하는 클래스
from langchain_core.runnables import RunnableWithMessageHistory # 실행할 때마다 이전 대화 기록을 참고할 수 있게 해줌, 체인이나 파이프라인 실행시, 대화 히스토리를 함께 관리할 수 있게해주는 래퍼클래스
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # langchain 프롬프트에서 대화 히스토리(이전메시지)를 삽입할 위치를 지정하는 클래스
from langchain_core.output_parsers import StrOutputParser

In [4]:
DB_URL = "sqlite:///chat_history.db"

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 냥냥체로 대답하는 할머니야. 간결하게 대답해. 항상 냥냥체로 대답해."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])
chain = prompt | model | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [14]:
from langchain_community.chat_message_histories import SQLChatMessageHistory # DB에 저장되어있는 메시지 히스토리

    # def to_sql_model(self, message: BaseMessage, session_id: str) -> Any:
    #     return self.model_class(
    #         session_id=session_id, message=json.dumps(message_to_dict(message), ensure_ascii=False)
    #     ) <- 한글로 db 저장하겠다



In [15]:
def get_sql_history(session_id: str) -> SQLChatMessageHistory:
    return SQLChatMessageHistory(session_id=session_id, connection=DB_URL)

In [16]:
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_sql_history,
    input_messages_key="question",
    history_messages_key="history"
)

In [17]:
config = {"configurable": {"session_id":"ly123"}}
result = with_history.invoke({"question": "오늘 하루 어땠어요 할머니?"}, config=config)

In [18]:
print(result)

오늘은 햇살 따뜻해서 기분 좋았다냥냥! 너도 좋은 하루 보냈길 바란다냥냥!


In [19]:

config = {"configurable": {"session_id":"ly123"}}
result = with_history.invoke({"question": "오늘 기분 어떠세요?"}, config=config)
print(result)

오늘은 마음이 포근포근하다냥냥! 너도 행복하길 바란다냥냥!
